In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from scipy import stats
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

In [4]:
df = pd.read_excel("/kaggle/input/datasets/ranatarekahmed1130/dataset-for-data-analytics/Dataset for Data Analytics.xlsx")

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())

Dataset Shape: (1200, 14)

First 5 Rows:
     OrderID       Date CustomerID  Product  Quantity  UnitPrice  \
0  ORD200000 2023-01-04     C72649  Monitor         5     570.62   
1  ORD200001 2024-08-23     C75739    Phone         2     151.35   
2  ORD200002 2024-02-27     C81728   Tablet         5     550.68   
3  ORD200003 2023-10-15     C33540    Chair         1     273.19   
4  ORD200004 2025-05-08     C81840  Printer         4     626.01   

  ShippingAddress PaymentMethod OrderStatus TrackingNumber  ItemsInCart  \
0     928 Main St    Debit Card     Shipped    TRK37947903            7   
1     823 Main St        Online     Shipped    TRK91186779            3   
2     512 Main St   Credit Card   Cancelled    TRK42903982            8   
3     275 Main St    Debit Card    Returned    TRK62788070            5   
4     668 Main St        Online   Delivered    TRK29241424            8   

  CouponCode ReferralSource  TotalPrice  
0     SAVE10      Instagram     2853.10  
1     SAVE10   

# EDA

In [5]:
print("\n====================")
print("MISSING VALUES")
print("====================")

print(df.isnull().sum())

print("\n====================")
print("SUMMARY STATISTICS")
print("====================")

print(df.describe())


MISSING VALUES
OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

SUMMARY STATISTICS
                      Date     Quantity    UnitPrice  ItemsInCart   TotalPrice
count                 1200  1200.000000  1200.000000  1200.000000  1200.000000
mean   2024-03-22 16:58:48     2.945833   356.412750     5.485000  1053.968300
min    2023-01-01 00:00:00     1.000000    11.390000     1.000000    11.390000
25%    2023-08-03 18:00:00     2.000000   186.062500     4.000000   410.520000
50%    2024-03-23 00:00:00     3.000000   364.210000     5.000000   823.615000
75%    2024-11-08 12:00:00     4.000000   521.570000     7.000000  1578.475000
max    2025-06-30 00:00:00     5.000000   699.930000    10.000000  3456.4000

In [7]:
# =====================================================
# HANDLE MISSING VALUES
# =====================================================

print("\n====================")
print("MISSING VALUE IMPUTATION")
print("====================")

# CouponCode is categorical
df["CouponCode"] = df["CouponCode"].fillna("NoCoupon")
print("After Handling Misssing Data")
print(df.isnull().sum())


MISSING VALUE IMPUTATION
After Handling Misssing Data
OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
dtype: int64


In [9]:
# =====================================================
# OUTLIER DETECTION USING Z-SCORE
# =====================================================
numeric_cols = [
    "Quantity",
    "UnitPrice",
    "ItemsInCart",
    "TotalPrice"
]
print("\n====================")
print("Z-SCORE OUTLIERS")
print("====================")

z_scores = np.abs(
    stats.zscore(df[numeric_cols])
)

outliers_z = (z_scores > 3).any(axis=1)

print(
    "Number of Outliers (Z-Score):",
    outliers_z.sum()
)

# Remove Outliers
df_z = df[~outliers_z]


Z-SCORE OUTLIERS
Number of Outliers (Z-Score): 0


In [12]:
# =====================================================
# OUTLIER DETECTION USING IQR
# =====================================================

print("\n====================")
print("IQR OUTLIERS")
print("====================")

df_iqr = df.copy()

for col in numeric_cols:

    Q1 = df_iqr[col].quantile(0.25)
    Q3 = df_iqr[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df_iqr = df_iqr[
        (df_iqr[col] >= lower)
        &
        (df_iqr[col] <= upper)
    ]

print(
    "Shape After IQR Removal:",
    df_iqr.shape
)




IQR OUTLIERS
Shape After IQR Removal: (1192, 14)


# Feature Engineering

In [15]:
# Convert Date
df["Date"] = pd.to_datetime(df["Date"])

# Feature 1
df["OrderMonth"] = df["Date"].dt.month

# Feature 2
df["OrderYear"] = df["Date"].dt.year

# Feature 3
df["AvgItemValue"] = (
    df["TotalPrice"] / df["Quantity"]
)

# Feature 4
df["CartEfficiency"] = (
    df["Quantity"] / df["ItemsInCart"]
)

# Feature 5
df["UsedCoupon"] = np.where(
    df["CouponCode"] == "NoCoupon",
    0,
    1
)
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice,OrderMonth,OrderYear,AvgItemValue,CartEfficiency,UsedCoupon
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10,1,2023,570.62,0.714286,1
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70,8,2024,151.35,0.666667,1
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40,2,2024,550.68,0.625000,1
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19,10,2023,273.19,0.200000,1
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04,5,2025,626.01,0.500000,1


# The mathematically clean dataset ready for machine learning algorithms

In [16]:
df.to_csv(
    "cleaned_feature_engineered_dataset.csv",
    index=False
)

print(
    "\nCleaned dataset saved successfully!"
)


Cleaned dataset saved successfully!
